In [55]:
from pathlib import Path
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT = Path.cwd()

if PROJECT.name.lower() == "notebook":
    PROJECT = PROJECT.parent

POLICY_DIR = PROJECT / "data" / "policies"

print("Project folder:", PROJECT)
print("Policy folder:", POLICY_DIR)

policy_files = sorted(POLICY_DIR.glob("*.txt"))

print("\nPolicy documents found:", len(policy_files))

for file in policy_files:
    print("-", file.name)

Project folder: c:\Users\kamali\OneDrive\Documents\captone
Policy folder: c:\Users\kamali\OneDrive\Documents\captone\data\policies

Policy documents found: 5
- delivery_policy.txt
- escalation_policy.txt
- payment_policy.txt
- privacy_policy.txt
- refund_policy.txt


In [56]:
chunks = []

CHUNK_SIZE = 40
OVERLAP = 10
STEP_SIZE = CHUNK_SIZE - OVERLAP

for file in policy_files:

    text = file.read_text(encoding="utf-8")

    words = text.split()

    start = 0

    while start < len(words):

        end = start + CHUNK_SIZE

        chunk_words = words[start:end]

        chunk_text = " ".join(chunk_words)

        chunks.append({
            "source": file.name,
            "text": chunk_text
        })

        if end >= len(words):
            break

        start += STEP_SIZE

print("Total chunks:", len(chunks))

Total chunks: 15


In [57]:
chunk_counts = Counter(
    chunk["source"] for chunk in chunks
)

for source, count in chunk_counts.items():
    print(source, ":", count, "chunks")

print("\nTotal chunks:", len(chunks))

assert len(chunks) == 15

expected_counts = {
    "refund_policy.txt": 4,
    "delivery_policy.txt": 3,
    "escalation_policy.txt": 3,
    "payment_policy.txt": 3,
    "privacy_policy.txt": 2
}

assert dict(chunk_counts) == expected_counts

print("\nAll chunk count checks passed!")

delivery_policy.txt : 3 chunks
escalation_policy.txt : 3 chunks
payment_policy.txt : 3 chunks
privacy_policy.txt : 2 chunks
refund_policy.txt : 4 chunks

Total chunks: 15

All chunk count checks passed!


In [58]:
# Extract chunk text
chunk_texts = [chunk["text"] for chunk in chunks]

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Fit on policy chunks
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (15, 200)


In [59]:
def retrieve(query, k=3):
    # Convert user query into TF-IDF
    query_vector = vectorizer.transform([query])

    # Calculate cosine similarity
    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Get top k matching chunk indexes
    top_indices = similarities.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks[index]["source"],
            "score": float(similarities[index]),
            "text": chunks[index]["text"]
        })

    return results

In [60]:
test_results = retrieve("refund policy late delivery", k=3)

for i, result in enumerate(test_results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"])


Result 1
Source: refund_policy.txt
Score: 0.2719
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Result 2
Source: refund_policy.txt
Score: 0.2514
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Result 3
Source: refund_policy.txt
Score: 0.2454
Text: late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original


TF-IDF — Notebook explanation
TF (Term Frequency): Oru word, particular chunk-la evlo frequent-ah varudhu-nu measure pannum.

IDF (Inverse Document Frequency): Ella chunks-layum common-ah irukkura words-ku less importance; rare and useful words-ku more importance kudukkum.

Limitation: TF-IDF exact word matching-based. Example, query-la refund irundha reimbursement maadhiri synonym words irundhaalum relevant chunk miss aagalam.

In [61]:
import sys
from pathlib import Path

PROJECT = Path.cwd()

if PROJECT.name.lower() == "notebook":
    PROJECT = PROJECT.parent

print("Project folder:", PROJECT)
print("mock_llm.py exists:", (PROJECT / "mock_llm.py").exists())

sys.path.insert(0, str(PROJECT))

Project folder: c:\Users\kamali\OneDrive\Documents\captone
mock_llm.py exists: True


In [62]:
import mock_llm

print("Mock LLM imported successfully!")

Mock LLM imported successfully!


In [63]:
def answer(question):
    # Retrieve top 3 relevant chunks
    results = retrieve(question, k=3)

    # Build context with source filenames
    context = "\n\n".join(
        f"[{item['source']}]\n{item['text']}"
        for item in results
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a POLICY ASSISTANT. "
                "Answer only using the supplied policy context. "
                "Do not invent information. "
                "If the answer is not available in the context, "
                "reply exactly: I could not find the answer in the supplied policies."
            )
        },
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{context}\n"
                f"QUESTION:\n{question}"
            )
        }
    ]

    response = mock_llm.chat(messages)

    answer_text = response.choices[0].message.content

    sources = list(dict.fromkeys(
        item["source"] for item in results
    ))

    return answer_text, sources

In [64]:
question = "What happens if an order is delivered after 45 minutes?"

rag_answer, sources = answer(question)

print("Question:", question)
print("\nAnswer:", rag_answer)
print("\nSources:", sources)

Question: What happens if an order is delivered after 45 minutes?

Answer: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.

Sources: ['refund_policy.txt', 'delivery_policy.txt']


In [65]:
EVAL_FILE = PROJECT / "data" / "raw" / "policy_eval.csv"

print("Evaluation file exists:", EVAL_FILE.exists())

Evaluation file exists: True


In [66]:
import pandas as pd

eval_df = pd.read_csv(EVAL_FILE)

print("Dataset shape:", eval_df.shape)
print("\nColumn names:")
print(eval_df.columns.tolist())

print("\nFirst 5 rows:")
display(eval_df.head())

Dataset shape: (14, 3)

Column names:
['question', 'expected_doc', 'expected_fact']

First 5 rows:


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more


In [67]:
evaluation_results = []

for _, row in eval_df.iterrows():

    question = row["question"]
    expected_doc = row["expected_doc"]
    expected_fact = row["expected_fact"]

    generated_answer, sources = answer(question)

    # Check whether expected document is retrieved
    doc_match = expected_doc in sources

    # Check whether expected fact appears in answer
    fact_match = (
        str(expected_fact).lower()
        in generated_answer.lower()
    )

    # Check fallback for unanswerable questions
    is_unanswerable = (
        pd.isna(expected_doc) or str(expected_doc).strip() == ""
    )

    fallback_match = (
        generated_answer.startswith("I could not find")
        if is_unanswerable
        else None
    )

    evaluation_results.append({
        "question": question,
        "expected_doc": expected_doc,
        "doc_match": doc_match,
        "fact_match": fact_match,
        "fallback_match": fallback_match,
        "answer": generated_answer
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Document retrieval matches:",
      evaluation_df["doc_match"].sum())

print("Expected fact matches:",
      evaluation_df["fact_match"].sum())

print("Unanswerable fallback checks:")
print(evaluation_df["fallback_match"].dropna().value_counts())

display(evaluation_df)

Document retrieval matches: 12
Expected fact matches: 12
Unanswerable fallback checks:
Series([], Name: count, dtype: int64)


,question,expected_doc,doc_match,fact_match,fallback_match,answer
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,None,"order is delivered in more than 45 minutes, th..."
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,None,A refund above INR 1000 must be approved by a ...
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,None,A refund request must be raised within 7 days ...
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,None,A delivery fee of INR 25 is charged on orders ...
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,None,"of INR 25 is charged on orders below INR 199, ..."
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,None,Electronics items are delivered only after the...
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,None,Cash on delivery is available only for orders ...
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,None,"If a payment fails but money is deducted, the ..."
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,None,Wallet balance can be used for any order and i...
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,None,Delivery addresses are deleted from active sys...


In [68]:
unanswerable_mask = (
    eval_df["expected_doc"]
    .fillna("")
    .str.strip()
    .str.upper()
    .isin(["NONE", ""])
)

fallback_results = evaluation_df.loc[
    unanswerable_mask, "answer"
].apply(
    lambda x: str(x).startswith("I could not find")
)

print("Unanswerable questions:", unanswerable_mask.sum())
print("Correct fallback answers:", fallback_results.sum())
print("Fallback accuracy:",
      f"{fallback_results.sum()}/{len(fallback_results)}")

assert fallback_results.all()

print("\nAll fallback checks passed!")

Unanswerable questions: 2
Correct fallback answers: 2
Fallback accuracy: 2/2

All fallback checks passed!


In [69]:
# Create chunks without overlap
chunks_no_overlap = []

CHUNK_SIZE = 40
OVERLAP = 0
STEP_SIZE = CHUNK_SIZE - OVERLAP

for file in policy_files:
    text = file.read_text(encoding="utf-8")
    words = text.split()

    for start in range(0, len(words), STEP_SIZE):
        end = start + CHUNK_SIZE

        chunk_text = " ".join(words[start:end])

        chunks_no_overlap.append({
            "source": file.name,
            "text": chunk_text
        })

print("Total zero-overlap chunks:", len(chunks_no_overlap))

Total zero-overlap chunks: 13


In [70]:
# Create separate TF-IDF model for zero-overlap chunks

zero_texts = [
    chunk["text"] for chunk in chunks_no_overlap
]

zero_vectorizer = TfidfVectorizer()

zero_tfidf_matrix = zero_vectorizer.fit_transform(zero_texts)

print("Zero-overlap TF-IDF matrix shape:",
      zero_tfidf_matrix.shape)


Zero-overlap TF-IDF matrix shape: (13, 200)


In [71]:
def retrieve_zero(query, k=3):

    query_vector = zero_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        zero_tfidf_matrix
    ).flatten()

    top_indices = similarities.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks_no_overlap[index]["source"],
            "score": float(similarities[index]),
            "text": chunks_no_overlap[index]["text"]
        })

    return results

In [72]:
test_zero = retrieve_zero(
    "refund policy late delivery",
    k=3
)

for i, result in enumerate(test_zero, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"])


Result 1
Source: refund_policy.txt
Score: 0.3866
Text: receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR

Result 2
Source: refund_policy.txt
Score: 0.2531
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Result 3
Source: delivery_policy.txt
Score: 0.1564
Text: Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,


In [73]:
def answer_zero(question):
    results = retrieve_zero(question, k=3)

    context = "\n\n".join(
        f"[{item['source']}]\n{item['text']}"
        for item in results
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a POLICY ASSISTANT. "
                "Answer only using the supplied context. "
                "Do not invent information. "
                "If the answer is unavailable, reply: "
                "I could not find the answer in the supplied policies."
            )
        },
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{context}\n"
                f"QUESTION:\n{question}"
            )
        }
    ]

    response = mock_llm.chat(messages)

    return response.choices[0].message.content

In [74]:
zero_eval_results = []

for _, row in eval_df.iterrows():

    question = row["question"]
    expected_doc = str(row["expected_doc"]).strip()
    expected_fact = str(row["expected_fact"]).strip()

    results = retrieve_zero(question, k=3)
    generated_answer = answer_zero(question)

    sources = [item["source"] for item in results]

    is_unanswerable = expected_doc.upper() == "NONE"

    zero_eval_results.append({
        "question": question,
        "expected_doc": expected_doc,
        "doc_match": (
            expected_doc in sources
            if not is_unanswerable else None
        ),
        "fact_match": (
            expected_fact.lower() in generated_answer.lower()
            if not is_unanswerable else None
        ),
        "fallback_match": (
            generated_answer.startswith("I could not find")
            if is_unanswerable else None
        ),
        "answer": generated_answer
    })

zero_evaluation_df = pd.DataFrame(zero_eval_results)

print("Document matches:",
      zero_evaluation_df["doc_match"].sum(), "/ 12")

print("Fact matches:",
      zero_evaluation_df["fact_match"].sum(), "/ 12")

print("Fallback matches:",
      zero_evaluation_df["fallback_match"].sum(), "/ 2")

display(zero_evaluation_df)

Document matches: 12 / 12
Fact matches: 10 / 12
Fallback matches: 2 / 2


,question,expected_doc,doc_match,fact_match,fallback_match,answer
0,What delivery time makes a completed order eli...,refund_policy.txt,True,False,None,receives a full refund of the order amount.
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,None,A refund above INR\n\n \n1000 must be approved...
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,None,A refund request must be raised within 7 days ...
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,None,A delivery fee of INR 25 is charged on orders ...
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,None,A delivery fee of INR 25 is charged on orders ...
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,None,Electronics items are delivered only after the...
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,None,Cash on delivery is available only for orders ...
7,How soon is money returned when a payment fail...,payment_policy.txt,True,False,None,"If a payment fails but money is deducted, the ..."
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,None,Wallet balance can be used for any order and i...
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,None,Delivery addresses are deleted from active sys...


In [75]:
for i, chunk in enumerate(chunks_no_overlap):
    if chunk["source"] in [
        "refund_policy.txt",
        "payment_policy.txt"
    ]:
        print(f"\nChunk {i} | {chunk['source']}")
        print(chunk["text"])


Chunk 5 | payment_policy.txt
Payment Policy. Customers can pay using UPI, debit card, credit card or cash on delivery. Cash on delivery is available only for orders up to INR 2500. If a payment fails but money is deducted, the amount is returned automatically

Chunk 6 | payment_policy.txt
within 48 hours. A customer who sees two charges for one order should raise a billing ticket with the order identifier. Wallet balance can be used for any order and is never refunded as cash. Coupons cannot be combined with

Chunk 7 | payment_policy.txt
wallet balance on the same order.

Chunk 10 | refund_policy.txt
Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Chunk 11 | refund_policy.txt
receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund i

### Retrieval and Grounding Evaluation

**With 10-word overlap:**
- Answerable questions: 12
- Correct document retrieval: 12/12
- Expected fact matches: 12/12
- Unanswerable fallback checks: 2/2

All evaluation checks passed.

**Without overlap:**
- Total chunks: 13
- Correct document retrieval: 12/12
- Expected fact matches: 10/12
- Unanswerable fallback checks: 2/2

Two questions failed fact matching because important sentences were split across chunk boundaries.

**Examples of chunk-boundary failures:**

1. Refund policy: Chunk 10 ended with "the customer", while Chunk 11 started with "receives a full refund of the order amount." The model did not receive the complete refund condition in its retrieved context.

2. Payment policy: Chunk 5 ended with "the amount is returned automatically", while Chunk 6 started with "within 48 hours." The response missed the expected time period.

**Conclusion:**

Using 10-word overlap improved fact matching from 10/12 to 12/12. Overlapping chunks preserve context across boundaries and improve retrieval-grounded answers. Therefore, 40-word chunks with 10-word overlap were selected for this project.

In [76]:
DB_PATH = PROJECT / "data" / "instamart.db"

print("Database exists:", DB_PATH.exists())

Database exists: True


In [77]:
import sqlite3

def get_order(order_id):
    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT order_id, status, order_amount, delivery_minutes
    FROM orders
    WHERE order_id = ?
    """

    result = conn.execute(query, (order_id,)).fetchone()
    conn.close()

    if result is None:
        return {"error": "order not found"}

    return {
        "order_id": result[0],
        "status": result[1],
        "order_amount": float(result[2]),
        "delivery_minutes": result[3]
    }

In [78]:
get_order("ORD-000001")

{'error': 'order not found'}

In [79]:
conn = sqlite3.connect(DB_PATH)

sample_orders = pd.read_sql_query(
    "SELECT order_id, status, order_amount, delivery_minutes FROM orders LIMIT 5",
    conn
)

conn.close()

display(sample_orders)

,order_id,status,order_amount,delivery_minutes
0,ORD-110093,completed,127.50,19.0
1,ORD-111396,completed,1598.00,25.0
2,ORD-102006,cancelled,21.75,NaN
3,ORD-105921,completed,86.40,33.0
4,ORD-108115,completed,104.40,21.0


In [80]:
SCENARIO_FILE = PROJECT / "data" / "raw" / "agent_scenarios.csv"

scenario_df = pd.read_csv(SCENARIO_FILE)

display(scenario_df)

,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


In [81]:
get_order("ORD-100048")

{'order_id': 'ORD-100048',
 'status': 'completed',
 'order_amount': 128.25,
 'delivery_minutes': 65.0}

In [82]:
for order_id in scenario_df["message"].str.extract(
    r"(ORD-\d{6})"
)[0]:
    print(order_id, ":", get_order(order_id))

ORD-100048 : {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65.0}
ORD-100067 : {'order_id': 'ORD-100067', 'status': 'completed', 'order_amount': 135.0, 'delivery_minutes': 59.0}
ORD-100145 : {'order_id': 'ORD-100145', 'status': 'completed', 'order_amount': 1999.0, 'delivery_minutes': 54.0}
ORD-100001 : {'order_id': 'ORD-100001', 'status': 'completed', 'order_amount': 109.65, 'delivery_minutes': 20.0}
ORD-100025 : {'order_id': 'ORD-100025', 'status': 'cancelled', 'order_amount': 45.0, 'delivery_minutes': None}
ORD-999999 : {'error': 'order not found'}
ORD-100048 : {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65.0}


In [83]:
def search_policy(query):
    results = retrieve(query, k=3)

    return {
        "results": results
    }

In [84]:
search_policy("refund for late delivery above 45 minutes")

{'results': [{'source': 'refund_policy.txt',
   'score': 0.40696482179452864,
   'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'},
  {'source': 'refund_policy.txt',
   'score': 0.2750614507767933,
   'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'},
  {'source': 'refund_policy.txt',
   'score': 0.2542038916962993,
   'text': 'late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original'}]}

In [85]:
def calculate_refund(order_amount, delivery_minutes):

    if delivery_minutes is None:
        return {
            "refund_inr": 0.0,
            "reason": "Delivery time unavailable"
        }

    if delivery_minutes > 45:
        refund_inr = float(order_amount)
    else:
        refund_inr = 0.0

    if refund_inr > 1000:
        return {
            "refund_inr": refund_inr,
            "action": "human_approval_required"
        }

    return {
        "refund_inr": refund_inr,
        "action": "automatic_refund"
    }

In [86]:
print("S1:", calculate_refund(128.25, 65))
print("S3:", calculate_refund(1999, 54))
print("S4:", calculate_refund(109.65, 20))

S1: {'refund_inr': 128.25, 'action': 'automatic_refund'}
S3: {'refund_inr': 1999.0, 'action': 'human_approval_required'}
S4: {'refund_inr': 0.0, 'action': 'automatic_refund'}


In [87]:
import json
from datetime import datetime
from pathlib import Path

LEDGER_PATH = PROJECT / "outputs" / "refund_ledger.jsonl"
LEDGER_PATH.parent.mkdir(parents=True, exist_ok=True)

def issue_refund(order_id, refund_inr):
    entry = {
        "timestamp": datetime.now().isoformat(),
        "tool": "issue_refund",
        "order_id": order_id,
        "refund_inr": float(refund_inr)
    }

    with open(LEDGER_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

    return {
        "status": "refund_issued",
        "order_id": order_id,
        "refund_inr": float(refund_inr)
    }

In [88]:
print(issue_refund("ORD-100048", 128.25))

{'status': 'refund_issued', 'order_id': 'ORD-100048', 'refund_inr': 128.25}


In [89]:
def escalate_to_human(order_id, refund_inr):
    entry = {
        "timestamp": datetime.now().isoformat(),
        "tool": "escalate_to_human",
        "order_id": order_id,
        "refund_inr": float(refund_inr),
        "status": "pending_human_approval"
    }

    with open(LEDGER_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

    return {
        "status": "escalated",
        "order_id": order_id,
        "refund_inr": float(refund_inr),
        "message": "Human approval required"
    }

In [90]:
print(escalate_to_human("ORD-100145", 1999))

{'status': 'escalated', 'order_id': 'ORD-100145', 'refund_inr': 1999.0, 'message': 'Human approval required'}


In [91]:
TOOLS_SPEC = [
    {
        "name": "get_order",
        "description": "Get order status, amount and delivery time from the database.",
        "parameters": {
            "order_id": "string"
        }
    },
    {
        "name": "search_policy",
        "description": "Search company policy documents for relevant rules.",
        "parameters": {
            "query": "string"
        }
    },
    {
        "name": "calculate_refund",
        "description": "Calculate refund amount and determine whether human approval is required.",
        "parameters": {
            "order_amount": "number",
            "delivery_minutes": "number or null"
        }
    },
    {
        "name": "issue_refund",
        "description": "Record an eligible automatic refund in the refund ledger.",
        "parameters": {
            "order_id": "string",
            "refund_inr": "number"
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Send a refund request for human approval and record it in the ledger.",
        "parameters": {
            "order_id": "string",
            "refund_inr": "number"
        }
    }
]

TOOL_MAP = {
    "get_order": get_order,
    "search_policy": search_policy,
    "calculate_refund": calculate_refund,
    "issue_refund": issue_refund,
    "escalate_to_human": escalate_to_human
}

print("Number of tools:", len(TOOLS_SPEC))
print("Tool names:", list(TOOL_MAP.keys()))

Number of tools: 5
Tool names: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human']


In [92]:
import inspect

print(inspect.signature(mock_llm.chat))
print(inspect.getsource(mock_llm.chat))

(messages, tools=None)
def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")



In [93]:
import inspect

print(inspect.getsource(mock_llm._agent))

def _agent(messages, tools):
    user = next(m["content"] for m in messages if m["role"] == "user")
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    call = lambda n, a: _resp(None, [{"id": f"call_{len(tool_msgs)+1}", "name": n, "arguments": a}], "tool_calls", messages)
    if "[LOOP]" in user:
        return call("search_policy", {"query": "refund policy"})
    if not tool_msgs:
        m = re.search(r"ORD-\d{6}", user)
        return call("get_order", {"order_id": m.group(0)}) if m else _resp("NEEDS_INFO: please share the order id.", messages=messages)
    last = tool_msgs[-1]; data = json.loads(last["content"]); name = last["name"]
    if name == "get_order":
        if "error" in data:
            return _resp("NOT_FOUND: no such order.", messages=messages)
        return call("search_policy", {"query": "refund policy late delivery"})
    order = json.loads(next(m for m in tool_msgs if m["name"] == "get_order")["content"])
    if name == "search_policy":
        if o

In [94]:
import json
import time

def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):
    start_time = time.monotonic()

    messages = [
        {
            "role": "system",
            "content": "AGENT: You are an Instamart refund support agent."
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    steps = []
    tokens_used = 0
    llm_calls = 0

    for iteration in range(max_iterations):

        # 1. Timeout guardrail
        if time.monotonic() - start_time >= time_limit_seconds:
            return {
                "status": "FALLBACK_TIMEOUT",
                "final_answer": "Human review required: agent timed out.",
                "steps": steps,
                "llm_calls": llm_calls,
                "tokens_used": tokens_used
            }

        # 2. Ask the mock LLM
        response = mock_llm.chat(messages, tools=TOOLS_SPEC)
        llm_calls += 1

        tokens_used += response.usage.total_tokens

        # 3. Token budget guardrail
        if tokens_used > token_limit:
            return {
                "status": "FALLBACK_TOKEN_LIMIT",
                "final_answer": "Human review required: token budget exceeded.",
                "steps": steps,
                "llm_calls": llm_calls,
                "tokens_used": tokens_used
            }

        message = response.choices[0].message

        # 4. Final answer
        if not message.tool_calls:
            final_answer = message.content

            return {
                "status": final_answer.split(":")[0],
                "final_answer": final_answer,
                "steps": steps,
                "llm_calls": llm_calls,
                "tokens_used": tokens_used
            }

        # 5. Process requested tool calls
        assistant_tool_calls = []

        for tool_call in message.tool_calls:

            tool_name = tool_call.name
            arguments = tool_call.arguments
            tool_id = tool_call.id

            assistant_tool_calls.append({
                "id": tool_id,
                "name": tool_name,
                "arguments": arguments
            })

        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": assistant_tool_calls
        })

        for tool_call in message.tool_calls:

            tool_name = tool_call.name
            arguments = tool_call.arguments
            tool_id = tool_call.id

            # Execute the requested Python tool
            observation = TOOL_MAP[tool_name](**arguments)

            steps.append({
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            messages.append({
                "role": "tool",
                "tool_call_id": tool_id,
                "name": tool_name,
                "content": json.dumps(observation)
            })

    # 6. Maximum iteration fallback
    return {
        "status": "FALLBACK_MAX_ITERATIONS",
        "final_answer": "Human review required: maximum iterations reached.",
        "steps": steps,
        "llm_calls": llm_calls,
        "tokens_used": tokens_used
    }

In [95]:
s1 = scenario_df.loc[
    scenario_df["scenario_id"] == "S1"
].iloc[0]

result_s1 = run_agent(
    user_message=s1["message"],
    scenario_id="S1"
)

print("Status:", result_s1["status"])
print("Final answer:", result_s1["final_answer"])
print("LLM calls:", result_s1["llm_calls"])
print("Tokens used:", result_s1["tokens_used"])

print("\nTool execution sequence:")
for step in result_s1["steps"]:
    print(step["tool"], "->", step["observation"])

Status: REFUND
Final answer: REFUND: INR 128.25 issued.
LLM calls: 5
Tokens used: 898

Tool execution sequence:
get_order -> {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65.0}
search_policy -> {'results': [{'source': 'refund_policy.txt', 'score': 0.2719281507693355, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.25137413448459545, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'refund_policy.txt', 'score': 0.24540580878362586, 'text': 'late-delivery refund is paid. Refunds up to INR 1000 are approved automati

In [96]:
agent_results = {"S1": result_s1}

for _, row in scenario_df.iterrows():
    scenario_id = row["scenario_id"]

    if scenario_id == "S1":
        continue

    result = run_agent(
        user_message=row["message"],
        scenario_id=scenario_id
    )

    agent_results[scenario_id] = result

    print(f"\n{scenario_id}")
    print("Status:", result["status"])
    print("Answer:", result["final_answer"])
    print("LLM calls:", result["llm_calls"])
    print("Tokens:", result["tokens_used"])


S2
Status: REFUND
Answer: REFUND: INR 135.0 issued.
LLM calls: 5
Tokens: 898

S3
Status: ESCALATE
Answer: ESCALATE: refund above INR 1000 sent to a human lead.
LLM calls: 5
Tokens: 890

S4
Status: DENIED
Answer: DENIED: delivery was within 45 minutes.
LLM calls: 4
Tokens: 600

S5
Status: DENIED
Answer: DENIED: only completed orders can be refunded.
LLM calls: 3
Tokens: 331

S6
Status: NOT_FOUND
Answer: NOT_FOUND: no such order.
LLM calls: 2
Tokens: 68

S7
Status: FALLBACK_MAX_ITERATIONS
Answer: Human review required: maximum iterations reached.
LLM calls: 6
Tokens: 2988


In [97]:
AGENT_RUNS_PATH = PROJECT / "outputs" / "agent_runs.jsonl"
AGENT_RUNS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(AGENT_RUNS_PATH, "w", encoding="utf-8") as f:
    for scenario_id, result in agent_results.items():

        record = {
            "scenario_id": scenario_id,
            "status": result["status"],
            "final_answer": result["final_answer"],
            "llm_calls": result["llm_calls"],
            "tokens_used": result["tokens_used"],
            "steps": result["steps"]
        }

        f.write(json.dumps(record) + "\n")

print("Saved agent runs:", len(agent_results))
print("File:", AGENT_RUNS_PATH)

Saved agent runs: 7
File: c:\Users\kamali\OneDrive\Documents\captone\outputs\agent_runs.jsonl


In [98]:
with open(AGENT_RUNS_PATH, "r", encoding="utf-8") as f:
    saved_runs = [json.loads(line) for line in f if line.strip()]

print("JSONL records:", len(saved_runs))
print("Scenario IDs:", [r["scenario_id"] for r in saved_runs])

assert len(saved_runs) == 7
assert len({r["scenario_id"] for r in saved_runs}) == 7

print("All 7 agent run records verified!")

JSONL records: 7
Scenario IDs: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7']
All 7 agent run records verified!


In [99]:
token_demo = run_agent(
    user_message=s1["message"],
    scenario_id="S1_TOKEN",
    token_limit=200
)

print("Status:", token_demo["status"])
print("Final answer:", token_demo["final_answer"])
print("LLM calls:", token_demo["llm_calls"])
print("Tokens used:", token_demo["tokens_used"])
print("Completed tool steps:", len(token_demo["steps"]))

assert token_demo["status"] == "FALLBACK_TOKEN_LIMIT"
print("Token limit test passed!")

Status: FALLBACK_TOKEN_LIMIT
Final answer: Human review required: token budget exceeded.
LLM calls: 3
Tokens used: 348
Completed tool steps: 2
Token limit test passed!


In [100]:
timeout_demo = run_agent(
    user_message=s1["message"],
    scenario_id="S1_TIMEOUT",
    time_limit_seconds=0
)

print("Status:", timeout_demo["status"])
print("Final answer:", timeout_demo["final_answer"])
print("LLM calls:", timeout_demo["llm_calls"])

assert timeout_demo["status"] == "FALLBACK_TIMEOUT"
assert timeout_demo["llm_calls"] == 0

print("Timeout test passed!")

Status: FALLBACK_TIMEOUT
Final answer: Human review required: agent timed out.
LLM calls: 0
Timeout test passed!


In [101]:
STATE_DIR = PROJECT / "state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(state):
    checkpoint_path = STATE_DIR / f"{state['run_id']}.json"

    with open(checkpoint_path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=4)

    return checkpoint_path

print("Checkpoint folder:", STATE_DIR)
print("Folder exists:", STATE_DIR.exists())

Checkpoint folder: c:\Users\kamali\OneDrive\Documents\captone\state
Folder exists: True


In [102]:
def continue_checkpointed_agent(state, simulate_crash_after=None):
    start_time = time.monotonic()

    max_iterations = state["max_iterations"]
    token_limit = state["token_limit"]
    time_limit_seconds = state["time_limit_seconds"]

    while state["llm_calls"] < max_iterations:

        # Timeout guardrail
        if time.monotonic() - start_time >= time_limit_seconds:
            state["status"] = "FALLBACK_TIMEOUT"
            state["final_answer"] = (
                "Human review required: agent timed out."
            )
            return state

        # Call mock LLM
        response = mock_llm.chat(
            state["messages"],
            tools=TOOLS_SPEC
        )

        state["llm_calls"] += 1
        state["tokens_used"] += response.usage.total_tokens

        # Token budget guardrail
        if state["tokens_used"] > token_limit:
            state["status"] = "FALLBACK_TOKEN_LIMIT"
            state["final_answer"] = (
                "Human review required: token budget exceeded."
            )
            return state

        message = response.choices[0].message

        # Final answer
        if not message.tool_calls:
            state["final_answer"] = message.content
            state["status"] = message.content.split(":")[0]
            save_checkpoint(state)
            return state

        # Add assistant tool-call message
        assistant_tool_calls = []

        for tc in message.tool_calls:
            assistant_tool_calls.append({
                "id": tc.id,
                "name": tc.name,
                "arguments": tc.arguments
            })

        state["messages"].append({
            "role": "assistant",
            "content": None,
            "tool_calls": assistant_tool_calls
        })

        # Execute each requested tool
        for tc in message.tool_calls:

            tool_name = tc.name
            arguments = tc.arguments

            observation = TOOL_MAP[tool_name](**arguments)

            state["steps"].append({
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            state["tools_executed"].append(tool_name)
            state["step_number"] += 1

            state["messages"].append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": tool_name,
                "content": json.dumps(observation)
            })

            # Save immediately after every tool observation
            save_checkpoint(state)

            # Simulate a crash after the requested number of tools
            if (
                simulate_crash_after is not None
                and state["step_number"] == simulate_crash_after
            ):
                raise RuntimeError(
                    "Simulated crash after checkpoint was saved."
                )

    state["status"] = "FALLBACK_MAX_ITERATIONS"
    state["final_answer"] = (
        "Human review required: maximum iterations reached."
    )

    save_checkpoint(state)
    return state


def start_checkpointed_agent(
    user_message,
    scenario_id,
    run_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10,
    simulate_crash_after=None
):
    state = {
        "run_id": run_id,
        "scenario_id": scenario_id,
        "messages": [
            {
                "role": "system",
                "content": "AGENT: You are an Instamart refund support agent."
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        "tokens_used": 0,
        "llm_calls": 0,
        "step_number": 0,
        "steps": [],
        "tools_executed": [],
        "max_iterations": max_iterations,
        "token_limit": token_limit,
        "time_limit_seconds": time_limit_seconds
    }

    save_checkpoint(state)

    return continue_checkpointed_agent(
        state,
        simulate_crash_after=simulate_crash_after
    )


def resume_agent(run_id):
    checkpoint_path = STATE_DIR / f"{run_id}.json"

    with open(checkpoint_path, "r", encoding="utf-8") as f:
        state = json.load(f)

    print("Resuming run:", run_id)
    print("Previously completed tools:", state["tools_executed"])

    return continue_checkpointed_agent(state)

In [103]:
try:
    crash_demo = start_checkpointed_agent(
        user_message=s1["message"],
        scenario_id="S1",
        run_id="S1_CRASH_DEMO",
        simulate_crash_after=2
    )

except RuntimeError as e:
    print("Crash simulation:", e)

Crash simulation: Simulated crash after checkpoint was saved.


In [104]:
checkpoint_path = STATE_DIR / "S1_CRASH_DEMO.json"

with open(checkpoint_path, "r", encoding="utf-8") as f:
    checkpoint = json.load(f)

print("Run ID:", checkpoint["run_id"])
print("Scenario:", checkpoint["scenario_id"])
print("Completed tools:", checkpoint["tools_executed"])
print("Step number:", checkpoint["step_number"])
print("Tokens used:", checkpoint["tokens_used"])

assert checkpoint["tools_executed"] == [
    "get_order",
    "search_policy"
]

print("Checkpoint verification passed!")

Run ID: S1_CRASH_DEMO
Scenario: S1
Completed tools: ['get_order', 'search_policy']
Step number: 2
Tokens used: 98
Checkpoint verification passed!


In [105]:
resumed_result = resume_agent("S1_CRASH_DEMO")

print("Final status:", resumed_result["status"])
print("Final answer:", resumed_result["final_answer"])
print("LLM calls:", resumed_result["llm_calls"])
print("Tokens used:", resumed_result["tokens_used"])

print("\nComplete tool execution sequence:")
print(resumed_result["tools_executed"])

Resuming run: S1_CRASH_DEMO
Previously completed tools: ['get_order', 'search_policy']
Final status: REFUND
Final answer: REFUND: INR 128.25 issued.
LLM calls: 5
Tokens used: 898

Complete tool execution sequence:
['get_order', 'search_policy', 'calculate_refund', 'issue_refund']


## Task 9 — ReAct vs Plan-and-Execute

### ReAct Approach

ReAct (Reason + Act + Observe) performs one action at a time. The LLM selects a tool, observes its result, and decides the next action based on that observation.

In this capstone, the S1 scenario required 5 LLM calls and 898 tokens. S5 required 3 calls and 331 tokens, while S6 required 2 calls and 68 tokens.

### Plan-and-Execute Approach

Plan-and-Execute creates a sequence of actions before executing them. It can reduce repeated reasoning when the workflow is predictable, but a fixed plan may not respond well to unexpected results.

### Comparison

ReAct is more flexible because it can respond to database observations, such as a cancelled order or a missing order ID. For example, S5 was correctly denied because the order was cancelled, and S6 returned NOT_FOUND because the order did not exist.

Plan-and-Execute may be more efficient for simple, predictable workflows, but it requires additional checks when actual tool results differ from expectations.

### Conclusion

For this refund support agent, ReAct is the better choice because refund decisions depend on actual order status, delivery time, policy retrieval, and approval limits. Its ability to adapt to unexpected tool observations makes it safer and more suitable for this capstone.

In [107]:
from pathlib import Path
import json

REFUND_LEDGER_PATH = PROJECT / "outputs" / "refund_ledger.jsonl"

print("Ledger path:", REFUND_LEDGER_PATH)
print("File exists:", REFUND_LEDGER_PATH.exists())

Ledger path: c:\Users\kamali\OneDrive\Documents\captone\outputs\refund_ledger.jsonl
File exists: True


In [108]:
with open(REFUND_LEDGER_PATH, "r", encoding="utf-8") as f:
    ledger = [json.loads(line) for line in f if line.strip()]

print("Refund ledger records:", len(ledger))

for row in ledger:
    print(row)

print("\nRefund ledger verification complete!")

Refund ledger records: 25
{'timestamp': '2026-11-05T01:02:09.510670', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-05T01:03:09.995751', 'tool': 'escalate_to_human', 'order_id': 'ORD-100145', 'refund_inr': 1999.0, 'status': 'pending_human_approval'}
{'timestamp': '2026-11-05T01:06:26.964103', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-05T01:06:26.993938', 'tool': 'escalate_to_human', 'order_id': 'ORD-100145', 'refund_inr': 1999.0, 'status': 'pending_human_approval'}
{'timestamp': '2026-11-05T01:07:46.552473', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-05T01:08:41.524858', 'tool': 'issue_refund', 'order_id': 'ORD-100067', 'refund_inr': 135.0}
{'timestamp': '2026-11-05T01:08:41.532505', 'tool': 'escalate_to_human', 'order_id': 'ORD-100145', 'refund_inr': 1999.0, 'status': 'pending_human_approval'}
{'timestamp': '2026-11-05T01:25:45.44853